# Establishing a database connection

In [1]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from urllib.parse import unquote
from getpass import getpass
server_name   = "localhost"
database_name = "Everyloop"

user_name = "sa"
pwd = getpass("Password:")
connection_string = f"DRIVER=ODBC Driver 18 for SQL Server;SERVER={server_name};UID={user_name};PWD={pwd};DATABASE={database_name};TrustServerCertificate=yes"
url_string        = URL.create("mssql+pyodbc", query={"odbc_connect": connection_string})

try:    
    engine = create_engine(url_string)
    with engine.connect() as connection:
        print(f'Successfully connected to {database_name}!')
except Exception as e:
    print('Error while connecting to database:\n')
    print(e)

Successfully connected to Everyloop!


# Setting up MetaData and declare tables

In [7]:
from sqlalchemy import MetaData, Table, Column, Integer, String, ForeignKey
metadata_obj = MetaData()

user_table = Table(
    "pythonUsers",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("username", String(30)),
    Column("password", String),
    Column("server_id", ForeignKey('pythonServers.id'), nullable=False)
)

server_table = Table(
    "pythonServers",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("servername", String),
    Column("IP", String(15))
)

for name, table in metadata_obj.tables.items():
    print(f"Table: {name}")

    for column in table.c:
        print(f"{column.name.ljust(15)}{column.type}")

    print()

Table: pythonUsers
id             INTEGER
username       VARCHAR(30)
password       VARCHAR
server_id      INTEGER

Table: pythonServers
id             INTEGER
servername     VARCHAR
IP             VARCHAR(15)



# Create all metadata tables

In [8]:
metadata_obj.create_all(engine)

print("Created tables:")

for table in metadata_obj.tables:
    print(table)

Created tables:
pythonUsers
pythonServers


# Drop all metadata tables

In [9]:
#metadata_obj.drop_all(engine)

print("Dropped tables:")

for table in metadata_obj.tables:
    print(table)

Dropped tables:
pythonUsers
pythonServers


# Insert server data (multiple rows)

In [10]:
from sqlalchemy import insert
import random, string

servers = [
    {"servername": "Mario", "IP": "192.168.0.23"},
    {"servername": "Luigi", "IP": "192.168.0.185"},
    {"servername": "Yoshi", "IP": "192.168.0.15"},
    {"servername": "Bowser", "IP": "192.168.0.97"},
]

for server in servers:
    print(f"{server['servername'].ljust(15)}{server['IP']}")

statement = insert(server_table)
print(statement)

with engine.begin() as conn:
    conn.execute(statement, servers)

Mario          192.168.0.23
Luigi          192.168.0.185
Yoshi          192.168.0.15
Bowser         192.168.0.97
INSERT INTO "pythonServers" (id, servername, "IP") VALUES (:id, :servername, :IP)


# Insert user data (single row)

In [11]:
from sqlalchemy import insert
import random, string

username = input('Enter username:')
password = ''.join(random.choice(string.ascii_letters + string.digits) for i in range(10))
server_id = random.randint(1, 4)

statement = insert(user_table).values(username=username, password=password, server_id=server_id)

print(statement)
print(statement.compile().params)

with engine.begin() as conn:
    result = conn.execute(statement)
    print(f"primary key value = {result.inserted_primary_key[0]}")

INSERT INTO "pythonUsers" (username, password, server_id) VALUES (:username, :password, :server_id)
{'username': 'sa', 'password': 's5leVUbtgR', 'server_id': 2}
primary key value = 6


# Select queries
In order to generate select queries we must first import select()

In [12]:
from sqlalchemy import select

Below are a few example of select queries:

### Select full table data

In [13]:
query = select(user_table)
print(query)

SELECT "pythonUsers".id, "pythonUsers".username, "pythonUsers".password, "pythonUsers".server_id 
FROM "pythonUsers"


### Projection

In [14]:
query = select(user_table.c.username, user_table.c.password, ("Hi " + user_table.c.username + "!").label('greeting'))
print(f"{query}\n\n{query.compile().params}")

SELECT "pythonUsers".username, "pythonUsers".password, :username_1 || "pythonUsers".username || :param_1 AS greeting 
FROM "pythonUsers"

{'username_1': 'Hi ', 'param_1': '!'}


### Selection

In [15]:
query = select(user_table).where(user_table.c.username == 'Fredrik')
print(f"{query}\n\n{query.compile().params}")

SELECT "pythonUsers".id, "pythonUsers".username, "pythonUsers".password, "pythonUsers".server_id 
FROM "pythonUsers" 
WHERE "pythonUsers".username = :username_1

{'username_1': 'Fredrik'}


### Order by

In [16]:
query = select(user_table).order_by(user_table.c.username.asc()).limit(3)
print(query)

SELECT "pythonUsers".id, "pythonUsers".username, "pythonUsers".password, "pythonUsers".server_id 
FROM "pythonUsers" ORDER BY "pythonUsers".username ASC
 LIMIT :param_1


### Joining tables

In [17]:
query = select(user_table.c.id, user_table.c.username, server_table.c.servername).join_from(user_table, server_table)
print(query)

SELECT "pythonUsers".id, "pythonUsers".username, "pythonServers".servername 
FROM "pythonUsers" JOIN "pythonServers" ON "pythonServers".id = "pythonUsers".server_id


### Grouping and aggregation

In [18]:
from sqlalchemy import func

query = (
    select(server_table.c.servername, server_table.c.IP, func.count(user_table.c.id).label('Number of users'))
    .join_from(user_table, server_table)
    .group_by(server_table.c.servername, server_table.c.IP)
)

print(query)

SELECT "pythonServers".servername, "pythonServers"."IP", count("pythonUsers".id) AS "Number of users" 
FROM "pythonUsers" JOIN "pythonServers" ON "pythonServers".id = "pythonUsers".server_id GROUP BY "pythonServers".servername, "pythonServers"."IP"


# Fetch data

In [19]:
print(f"{query}\n\n{query.compile().params}\n")

with engine.connect() as conn:
    result = conn.execute(query)
    
    for column_name in result.keys():
        print(column_name.upper().ljust(20), end=' ')
    
    print()

    for row in result:
        for field in row:
            print(str(field).ljust(20), end=' ')
        
        print()

SELECT "pythonServers".servername, "pythonServers"."IP", count("pythonUsers".id) AS "Number of users" 
FROM "pythonUsers" JOIN "pythonServers" ON "pythonServers".id = "pythonUsers".server_id GROUP BY "pythonServers".servername, "pythonServers"."IP"

{}

SERVERNAME           IP                   NUMBER OF USERS      
Luigi                192.168.0.185        2                    
Mario                192.168.0.23         1                    
Bowser               192.168.0.97         3                    
